<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 10 · Linear and Generalized Linear Models for Return Prediction

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
Chapter 10 focuses on linear/GLM baselines. Here we:
- engineer cross-sectional features,
- fit ridge/lasso/logistic models with scikit-learn,
- evaluate IC/AUC metrics, and
- translate predictions into portfolio weights.


### Getting Help While Modeling
- **Appendix B** for pandas feature engineering patterns.
- **Appendix C** for scikit-learn estimator syntax.
- **Chapter 8** for performance metric definitions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_squared_error, roc_auc_score

## 1. Build Cross-Sectional Feature Panel

This section reshapes the time‑series price panel into a stacked (date, asset) structure with momentum and volatility features for every asset‑date pair.

### 1.0 Data Loading
We load prices once for the cross-sectional panel.

In [ ]:
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()

In [ ]:
panel = prices.ffill().pct_change()
features = panel.rolling(20).mean()
vol = panel.rolling(20).std()
X = pd.concat({"mom": features, "vol": vol}, axis=1).stack().dropna()
y = panel.shift(-1).stack().reindex(X.index).dropna()
X = X.loc[y.index]
data = pd.DataFrame({"mom": X["mom"], "vol": X["vol"], "label": y})
data.head()

### 1.1 Train/Test Split by Date

We cut the panel into an early training segment and a later test segment along the calendar axis to mimic a realistic research workflow.

In [ ]:
dates = data.index.get_level_values(0)
split_idx = int(len(dates) * 0.7)
split_date = dates.sort_values()[split_idx]
train = data.loc[dates <= split_date]
test = data.loc[dates > split_date]
X_train, y_train = train[["mom", "vol"]], train["label"]
X_test, y_test = test[["mom", "vol"]], test["label"]

## 2. Ridge vs. Lasso Regression

We compare ridge and lasso models on the same feature set to see how L2 vs. L1 regularization affects fit quality.

In [ ]:
ridge = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=5.0)),
])
lasso = Pipeline([
    ("scale", StandardScaler()),
    ("model", Lasso(alpha=0.001)),
])
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
lasso_pred = lasso.predict(X_test)
pd.Series({"ridge_mse": mean_squared_error(y_test, ridge_pred), "lasso_mse":
mean_squared_error(y_test, lasso_pred)})

## 3. Logistic Regression for Directional Bets

Here we reframe the prediction target as up/down moves and fit a logistic regression that outputs probabilities instead of raw returns.

In [ ]:
y_train_cls = (y_train > 0).astype(int)
y_test_cls = (y_test > 0).astype(int)
logit = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
logit.fit(X_train, y_train_cls)
logit_prob = logit.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test_cls, logit_prob)
auc

### Performance Helper
We reuse the Chapter 8 metric utility here for completeness.

In [ ]:
def performance_stats(returns: pd.Series, risk_free: float = 0.02):
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - risk_free) / ann_vol if ann_vol > 0 else np.nan
    wealth = (1 + returns).cumprod()
    max_dd = (wealth / wealth.cummax() - 1).min()
    return pd.Series(
        {
            "annualized_return": ann_ret,
            "annualized_vol": ann_vol,
            "sharpe": sharpe,
            "max_drawdown": max_dd,
        }
    )

### 3.1 Portfolio Signal from Probabilities

We turn predicted probabilities into position sizes and evaluate the resulting long/short strategy using the performance helper.

In [ ]:
weights = pd.Series(logit_prob - 0.5, index=y_test_cls.index).clip(-0.5, 0.5)
asset_returns = pd.Series(y_test.values, index=y_test.index)
strategy = weights * asset_returns
performance_stats(strategy)

## 4. Exercises
### Exercise 1 – Elastic Net
Add an Elastic Net model to the comparison table.
<details><summary>Hint</summary>
Use <code>sklearn.linear_model.ElasticNet</code> with <code>l1_ratio</code> grid search.
</details>

### Exercise 2 – Feature Scaling Variants
Experiment with <code>RobustScaler</code> and compare coefficients.
<details><summary>Hint</summary>
Swap the scaler step inside the pipeline and refit models.
</details>

### Exercise 3 – Threshold Tuning
Test different probability thresholds (0.4/0.6, etc.) when generating portfolio weights.
<details><summary>Hint</summary>
Change the subtraction from 0.5 to the chosen cutoffs and recompute <code>performance_stats</code>.
</details>


## 5. Takeaways for Chapter 10
- Feature stacking turns time-series data into cross-sectional ML inputs.
- Pipelines with scalers keep training reproducible.
- Logistic probabilities make it easy to translate classification outputs into weights.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">